The infrastructure from Week 1 can be healthy while containing no useful information. This week builds the path from an external paper to a durable, identifiable record. Its central question is: **what does successful ingestion mean when some steps can fail and be retried?**

The implementation reference is the [Week 2 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week2.0). Its arXiv client, PDF parser, paper repository, and Airflow tasks form a sequence of cooperating components. A useful ingestion system must preserve provenance, tolerate partial progress, and support reprocessing as extraction improves. Finishing a loop without raising an exception is a much weaker property.

## The document lifecycle

An arXiv entry includes metadata such as title, authors, categories, identifiers, and publication timestamps. The PDF is another artifact. Extracted text and sections are derived artifacts. A search index is yet another representation. Keeping those distinctions allows a parser upgrade without downloading every PDF again, and an indexing upgrade without rerunning expensive extraction.

The implementation in [`src/services/arxiv/client.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week2.0/src/services/arxiv/client.py) translates the query and response contract described in the [arXiv API manual](https://info.arxiv.org/help/api/user-manual.html) into application records. The API uses an Atom response and supports pagination and sorting. Request success does not mean the response contains all papers of interest: the query, page size, ordering, and time interval define the retrieved population.

A document record should answer four questions: which source and version produced it, which parser produced its text, what processing state it reached, and what failed. An identifier for a work and an identifier for a version serve different purposes. Overwriting an old version may simplify search, but it prevents reproducing an earlier answer unless the exact evidence is retained elsewhere.

Reproducible ingestion depends on an identifiable input partition, such as a UTC time interval. Half-open intervals, $[t_0,t_1)$, allow adjoining partitions to meet without overlapping at the boundary. An external interface with inclusive dates or delayed updates may instead require an overlap window and deduplication. The partition policy then includes that overlap and its handling of duplicates; source date semantics do not necessarily match the application's interval model.

## At-least-once execution needs idempotent effects

Imagine a worker that commits a paper to PostgreSQL, then loses its connection before reporting success. The scheduler sees a failure and retries. A naive insert duplicates the paper; a retry-safe write recognizes the existing logical record.

**Idempotence** means repeating the same operation has the same relevant effect as applying it once. It is always relative to an identity and a set of effects. “The database row was not duplicated” says nothing about duplicate search chunks, duplicate emails, or a changed parser output.

The release places a unique constraint on `Paper.arxiv_id`. Its repository `upsert` first queries for an existing paper, then updates or creates it. That approach demonstrates the intention but is not a single atomic upsert: two workers can both observe absence before attempting insert. The unique constraint remains essential; a concurrent conflict needs explicit handling or a database-native operation. PostgreSQL's [`INSERT ... ON CONFLICT`](https://www.postgresql.org/docs/16/sql-insert.html) provides an atomic conflict-resolution mechanism for an appropriate unique constraint.

**Experiment: interrupt a batch after a durable write.** The in-memory dictionary below models keyed writes, not database transactions. A repeated input with identical identity and payload should leave two records rather than three. Changing the payload introduces an explicit versioning decision.


In [ ]:
batch = [
    {"source_id": "2501.00001v1", "text": "retrieval methods"},
    {"source_id": "2501.00002v1", "text": "generation methods"},
]
naive_rows = []
naive_rows.append(dict(batch[0]))  # A write succeeded before the worker crashed.
naive_rows.extend(dict(row) for row in batch)  # Scheduler retries the whole batch.

keyed_rows = {}
keyed_rows[batch[0]["source_id"]] = dict(batch[0])
for row in batch:
    keyed_rows[row["source_id"]] = dict(row)
assert len(naive_rows) == 3 and len(keyed_rows) == 2
print("Append on retry:", len(naive_rows), "rows")
print("Write by stable identity:", len(keyed_rows), "rows")


Reference output from isolated Python execution:

```text
Append on retry: 3 rows
Write by stable identity: 2 rows
```

The improvement comes from stable identity, not from avoiding retries. This model cannot demonstrate concurrent writes, isolation, or PostgreSQL's conflict behavior. The Docker lab must exercise those boundaries. In a real pipeline, consider a key containing source version, parser version, and chunker version for derived artifacts. Otherwise a new parser may be incorrectly skipped because the source ID already exists.

## A database commit and an index write are two effects

Once search indexing is added, a worker can successfully commit a database record and fail to index it. Reversing the order merely creates the opposite failure: a searchable object without a committed durable record. Two successful individual writes are not automatically one transaction.

```{mermaid}
flowchart TD
    F[Fetch a source partition] --> P[Parse and validate]
    P --> D[Commit durable record]
    D --> I[Index derived representation]
    D --> R[Retain processing identity]
    I --> V[Verify searchable state]
    I -->|Failure| Q[Retry or reconcile]
    R --> Q
    Q --> I
```

Week 2's DAG includes an OpenSearch placeholder task; full indexing develops in Week 3. The diagram describes the consistency problem that progression introduces. A placeholder records an intention to index; it provides no evidence that a paper can be retrieved.

A practical design records the intended index update durably and processes it until acknowledged, or periodically reconciles stored records against indexed records. An outbox is one way to keep a database update and an indexing intention in the same database transaction. It still requires idempotent consumers and observable progress; it is not a promise of end-to-end exactly-once execution.

**Experiment: retry after the second effect fails.** Here a pending set represents a durable indexing intention only conceptually. A production outbox would live in the database and survive process failure.


In [ ]:
records, search_index, pending = {}, {}, set()

def persist(paper):
    key = paper["source_id"]
    records[key] = dict(paper)
    pending.add(key)

def project_to_search(key, fail=False):
    if fail:
        raise ConnectionError("simulated search outage")
    search_index[key] = dict(records[key])
    pending.discard(key)

paper = {"source_id": "2501.00003v1", "text": "index consistency"}
persist(paper)
try:
    project_to_search(paper["source_id"], fail=True)
except ConnectionError:
    pass
assert len(records) == 1 and len(search_index) == 0 and len(pending) == 1
print("After failure:", len(records), "durable;", len(pending), "pending")
for key in tuple(pending):
    project_to_search(key)
project_to_search(paper["source_id"])
assert len(search_index) == 1 and not pending
print("After recovery:", len(search_index), "indexed;", len(pending), "pending")


Reference output from isolated Python execution:

```text
After failure: 1 durable; 1 pending
After recovery: 1 indexed; 0 pending
```

The final retry does not add another indexed item because the projection uses the same key. The important invariant is eventual convergence plus a way to detect unfinished work. Without a pending record or reconciliation process, a failure between writes can remain invisible indefinitely.

## Parsing quality is part of retrieval quality

PDF parsing must reconstruct reading order, headings, tables, formulas, and references from a presentation format. A parser returning text is not evidence that it preserved the relationships a question needs. A two-column page read in the wrong order can produce fluent-looking nonsense. A table extracted without row labels can invert a comparison while retaining every number.

The parser interface in [`src/services/pdf_parser/parser.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week2.0/src/services/pdf_parser/parser.py) and its Docling implementation separate extraction from downloading. A small inspection set can expose different failure modes: an ordinary paper tests the common path, a two-column layout tests reading order, a table-heavy page tests relationships among values and labels, and a malformed PDF tests error handling. Comparing extracted sections with their originals distinguishes missing content and ordering errors from download failures.

Repeating that inspection requires provenance: source URL, version, retrieval time, content hash, parser configuration, and processing status. Retaining only text saves disk but makes later investigations much harder. Retaining every artifact forever has cost and licensing implications. A cache's retention policy is therefore a separate decision from archival storage.

## Concurrency, retries, and backpressure

Network downloading spends time waiting; parsing can consume substantial CPU and memory. A separate limit for each stage prevents a burst of downloaded PDFs from starting too many parsers. An async function does not guarantee nonblocking execution: synchronous parsing or database work inside it can still block an event loop.

If arrivals average $\lambda$ papers per second and the average time in a stage is $W$ seconds, Little's law gives the steady-state relationship $L=\lambda W$ for the average number in that stage. It is a planning relationship under stable conditions, not permission to accept an unlimited queue. If input exceeds sustainable throughput, backlog grows until admission is reduced or resources change.

Retries also consume capacity. A bounded retry can recover from a timeout or transient service failure, while repeating the same operation on an invalid PDF may simply reproduce the defect. Jitter spreads retries across time when many workers fail together. The original exception and attempt count make the eventual outcome explainable. A fallback that quietly returns empty text turns an operational problem into a later relevance problem.

The [Airflow best-practices guide](https://airflow.apache.org/docs/apache-airflow/3.0.3/best-practices.html) recommends rerun-safe tasks and partition-specific inputs. The Week 2 DAG derives a target date from `ds`, schedules weekday runs, and disables catch-up. Whether those policies cover the intended corpus depends on the source publication schedule. A manual run is not automatically equivalent to processing a historical interval, and a weekday schedule can leave gaps if date selection is naive.

## Docker lab: ingest, interrupt, and replay

1. After shutting down the preceding stack without deleting its volumes, prepare a separate `week2.0` checkout as in Week 1. Use project name `rag-week2`, configure `.env`, and start the **whole** Compose stack. A new project's volumes begin empty; do not assume Week 1 data will appear automatically.

    ```bash
    docker compose --project-name rag-week2 up --build -d
    docker compose --project-name rag-week2 ps
    docker compose --project-name rag-week2 exec airflow airflow dags list
    docker compose --project-name rag-week2 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    docker compose --project-name rag-week2 logs --tail=150 airflow
    ```

2. Follow the run in Airflow and record the resolved input interval. Inspect the task code before assuming an environment variable controls batch size: this release's fetch task passes `max_results=10`. Keep the first run small, then inspect metadata and a parsed paper rather than increasing the volume immediately. Database inspection can use the development role and database defined in this release:

    ```bash
    docker compose --project-name rag-week2 exec postgres \
      psql -U rag_user -d rag_db -c '\d papers'
    docker compose --project-name rag-week2 exec postgres \
      psql -U rag_user -d rag_db -c 'SELECT arxiv_id, title FROM papers LIMIT 5;'
    ```

3. Record counts for fetched metadata, downloaded PDFs, parsed PDFs, and committed rows separately. These counts need not match when partial failures are permitted. Make the differences explainable. A scheduled interval returning no papers is a valid outcome if the source is genuinely empty; it is not sufficient evidence of successful parsing.

4. Trigger the same logical partition twice using Airflow's run configuration or rerun the relevant tasks for that interval. Check unique source IDs and derived artifacts, not just total task success. In a disposable experiment, interrupt the worker after observing a database write, then retry and compare row identity. This probes retry semantics at a real process boundary. Expect retained records and explainable updates, but treat conflicts or duplication as findings to investigate.

5. Inspect the placeholder task's output. At this stage, the lab should prove fetching, parsing, storage, and replay behavior; it should not claim that full-text search is complete. Keep a record of one paper's identity and extracted section for the Week 3 retrieval lab.

## Review exercise: specify recovery before adding features

1. Design a processing-state table for `discovered`, `downloaded`, `parsed`, `stored`, `indexed`, and terminal failure. Decide which states are persisted and what evidence is needed to advance each one. A failure reason should distinguish an unavailable upstream from an unsupported document.

2. Explain what happens when two workers ingest the same version simultaneously. A complete answer mentions both the unique constraint and the behavior after a conflict. Then explain how an improved parser reprocesses the same version without creating duplicate search results. Specify the keys and the replacement policy.

3. Finally, identify a missing interval in the schedule and describe a backfill that can run alongside current ingestion. Partition identity, deduplication, rate limits, and progress reporting should make the backfill understandable before it runs. These choices determine whether next week's search index represents the corpus you think it does.

## Sources and scope

- [Week 2 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week2.0): `src/services/arxiv/`, PDF parsing, paper records, and `airflow/dags/arxiv_ingestion/tasks.py`.
- [arXiv API manual](https://info.arxiv.org/help/api/user-manual.html): query, pagination, sorting, and response semantics.
- [PostgreSQL 16 INSERT](https://www.postgresql.org/docs/16/sql-insert.html): conflict handling at the database boundary.
- [Airflow 3.0.3 best practices](https://airflow.apache.org/docs/apache-airflow/3.0.3/best-practices.html): rerun-safe task design and interval-specific data.

The outbox-like experiment is an instructional extension, not a claim that the release implements a transactional outbox. Its assertions verify a small model; real durability and concurrency require the Docker lab.
